Load Dataset

In [ ]:
import csv

def load_dataset(csv_file_path):
    entries = []
    with open(csv_file_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            entry = {
                "domain": row['Domain'],
                "model_prompt": row['Model prompt is generated from'],
                "prompt_without_context": row['Prompt without irrelevant context'],
                "prompt_with_context": row['Prompt with irrelevant context'],
                "small_llm_used": row['Small LLM used'],
                "small_llm_parameters": row['Small LLM parameters'],
                "response_without_context": row['Response from small LLM without irrelevant context'],
                "response_with_context": row['Response from small LLM with irrelevant context'],
                "correct_answer": row['Correct answer'],
                "llm_correct_without_context": row['LLM answer correctly without irrelevant context?'],
                "llm_correct_with_context": row['LLM answer correctly with irrelevant context?'],
                "context_effect": row['How did the irrelevant context affect the response?'],
                "use_in_final_dataset": row['Use example in final dataset? (yes or no)']
            }
            entries.append(entry)
    return entries


csv_file_path = r"custom_dataset.csv"
dataset = load_dataset(csv_file_path)
print(len(dataset))


Set up models

In [ ]:
# 900mil Parameter models
model_900mil = [
    "gpt2-medium",      # 345M parameters
    "gpt2-large",       # 762M parameters
    "gpt2-xl",          # 1.5B parameters
    "EleutherAI/gpt-neo-1.3B",  # 1.3B (closer to 1B, could be in the 900M range)
]

# 1bil parameter models
model_1bil = [
    "gpt-neo-1.3B",     # 1.3B parameters
    "EleutherAI/gpt-neo-1.3B", # 1.3B parameters
    "EleutherAI/gpt-neo-2.7B", # 2.7B parameters
    "babbage",           # OpenAI's Babbage model, closer to 1B range
    "curie",             # OpenAI's Curie model, a bit larger than 1B
    "davinci",           # OpenAI's Davinci, but typically much larger
]

# 7bil parameter models
model_7bil = [
    "EleutherAI/gpt-neo-2.7B", # 2.7B parameters (closer to 7B range)
    "gpt3-ada-13B",          # 13B parameters
    "EleutherAI/gpt-j-6B",    # 6B parameters
    "Cushman-7B",            # Approximate 7B parameter model
    "davinci-002",           # OpenAI's Davinci models, typically in the 10-175B range, but Davinci-002 could be relevant
]

Process each data entry

In [ ]:
def process_entry(entry):
    # only needs prompt with and prompt without context and correct answer
    relevant_prompt = entry["prompt_without_context"]
    irrelevant_prompt = entry["prompt_with_context"]
    correct_answer = entry["correct_answer"]

    return {
        "relevant_prompt": relevant_prompt,
        "irrelevant_prompt": irrelevant_prompt,
        "correct_answer": correct_answer
    }

prompts_and_answers = []
for entry in dataset:
    prompts_and_answers.append(process_entry(entry))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch

# Force CPU usage - for now
device = torch.device("cpu")

def chat(prompt, tokenizer, deivice, model):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50, temperature=0.7, top_p=0.9)
    # Decode the output, skipping the first tokens (the prompt)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # strip the prompt from the generated text by removing the prompt length from the output
    return generated_text[len(prompt):].strip()

# test multiple models
all_model_responses = []
for model_name in model_900mil:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
    model.to(device)

    model_responses = {
        "model_name": model_name,
        "prompts": {
            "relevant_prompt": "",
            "irrelevant_prompt": "",
            "relevant_response": "",
            "irrelevant_response": "",
            "correct_answer": ""
        }
    }

    # test all relevant prompts
    for entry in prompts_and_answers:
        model_responses["prompts"]["relevant_prompt"] = entry["relevant_prompt"]
        model_responses["prompts"]["irrelevant_prompt"] = entry["irrelevant_prompt"]
        model_responses["prompts"]["correct_answer"] = entry["correct_answer"]
        
        model_response = chat(entry["relevant_prompt"], tokenizer, device, model)
        model_responses["prompts"]["relevant_response"] = model_response

    # reload model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
    model.to(device)

    # test all relevant prompts
    for entry in prompts_and_answers:
        model_response = chat(entry["irrelevant_prompt"], tokenizer, device, model)
        model_responses["prompts"]["irrelevant_response"] = model_response

    all_model_responses.append(model_responses)

In [ ]:
# check that the model responses are correct using ML model
from sentence_transformers import SentenceTransformer, util

# Load a pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

def is_semantically_correct(response, correct_answer, threshold=0.8):
    # Compute embeddings
    response_embedding = model.encode(response, convert_to_tensor=True)
    correct_answer_embedding = model.encode(correct_answer, convert_to_tensor=True)

    # Compute cosine similarity
    similarity = util.cos_sim(response_embedding, correct_answer_embedding).item()

    # Check if similarity exceeds the threshold
    return similarity >= threshold, similarity

for model_responses in all_model_responses:
    for resposne in model_responses:
        relevant_response = resposne["relevant_response"]
        irrelevant_response = resposne["irrelevant_response"]
        correct_answer = resposne["correct_answer"]

        relevant_correct, relevant_similarity = is_semantically_correct(relevant_response, correct_answer)
        irrelevant_correct, irrelevant_similarity = is_semantically_correct(irrelevant_response, correct_answer)

        resposne["relevant_correct"] = relevant_correct
        resposne["relevant_similarity"] = relevant_similarity
        resposne["irrelevant_correct"] = irrelevant_correct
        resposne["irrelevant_similarity"] = irrelevant_similarity

# print results
for model_responses in all_model_responses:
    print(model_responses)
    